# OSA Big Drop — day-on-day detector QC

Smoke-test the `osa_big_drop.big_drop` day-on-day detector.


In [ ]:
# --- QC parameters -----------------------------------------------------------
ANALYSIS_DATE_STR = '2026-07-26'   # last day of the week (today)
CLIENT = 'eastofengland'            # or None for base defaults
QC_STORE_IDS = [155]                # set to None for all stores

import datetime
analysis_date = datetime.date.fromisoformat(ANALYSIS_DATE_STR)
THIS_WEEK_END = str(analysis_date)
THIS_WEEK_START = str(analysis_date - datetime.timedelta(days=6))
LAST_WEEK_END = str(analysis_date - datetime.timedelta(days=7))
LAST_WEEK_START = str(analysis_date - datetime.timedelta(days=13))
LOOKBACK_START = str(analysis_date - datetime.timedelta(days=13))

print(f'Analysis date: {THIS_WEEK_END}')
print(f'Comparing today ({THIS_WEEK_END}) vs yesterday')

In [ ]:
import sys, os, logging, warnings
_root = os.path.abspath(os.path.join('..', '..', '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

logging.getLogger('snowflake.connector').setLevel(logging.ERROR)
logging.getLogger('snowflake.snowpark').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='snowflake')

from store_insight.pipeline.config.loader import load_config
from store_insight.pipeline.pipeline.session import create_session

cfg = load_config(CLIENT)
session = create_session(cfg)

print(f'Config loaded: client={cfg.client_name}')

In [ ]:
from snowflake.snowpark import functions as F

OSA_TABLE = cfg.tables.osa
HIERARCHY_TABLE = cfg.tables.hierarchy

osa_raw = (
    session.table(OSA_TABLE)
    .filter(F.col('DATE') >= LOOKBACK_START)
    .filter(F.col('DATE') <= THIS_WEEK_END)
    .select(
        F.col('STORE_ID'),
        F.col('ITEM_CODE'),
        F.col('ACTUAL_SALES_VOLUME'),
        F.col('LOST_SALES_VOLUME'),
        F.col('LOST_SALES_VALUE'),
        F.col('DATE'),
        F.col('INVENTORY_QTY'),
        F.col('DEPARTMENT_ID'),
        F.col('REPORTING_PERIOD_ID'),
    )
)

item = (
    session.table(HIERARCHY_TABLE)
    .select(
        F.col('ITEM_CODE'),
        F.col('ITEM_DESCRIPTION'),
        F.col('ITEM_HIERARCHY_1'),
        F.col('ITEM_HIERARCHY_2'),
        F.col('ITEM_HIERARCHY_3'),
    )
    .distinct()
)

osa = (
    osa_raw
    .join(item, on=['ITEM_CODE'], how='left')
    .filter(F.col('ITEM_HIERARCHY_2').is_not_null())
    .cache_result()
)

print(f'OSA rows loaded & cached: {osa.count():,}')

In [ ]:
from snowflake.snowpark import functions as F

# Check which sales days are actually in the loaded data for the selected store(s)
_earliest = str(datetime.date.fromisoformat(THIS_WEEK_END) - datetime.timedelta(days=7))
_check_data = osa.filter(F.col('STORE_ID').isin(QC_STORE_IDS)) if QC_STORE_IDS else osa

available = (
    _check_data
    .select(F.col('DATE'))
    .distinct()
    .filter(F.col('DATE') >= _earliest)
    .filter(F.col('DATE') <= THIS_WEEK_END)
    .sort(F.col('DATE').desc())
    .collect()
)

print('Available sales days in data for selected store(s):')
for row in available:
    print(' ', row['DATE'])

print()
print('BIG_DROP will compare:')
print('  today     =', THIS_WEEK_END)
if len(available) >= 2:
    print('  yesterday =', available[1]['DATE'])
else:
    print('  (no earlier day found in data)')


In [ ]:
from store_insight.pipeline.stages._1_detect.detect import detect_signals
from store_insight.pipeline.pipeline.config import DetectorConfig

det_cfg = DetectorConfig.from_config(cfg.detector_config.model_dump() if cfg.detector_config else {})
data = osa.filter(F.col('STORE_ID').isin(QC_STORE_IDS)) if QC_STORE_IDS else osa

signals = detect_signals(
    data,
    last_week_start=LAST_WEEK_START,
    last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START,
    this_week_end=THIS_WEEK_END,
    lookback_start=LOOKBACK_START,
    store_id=QC_STORE_IDS,
    config=det_cfg,
    cfg={'detectors': {'enabled': ['osa_big_drop.big_drop']}},
)

print(f'Total BIG_DROP signals: {len(signals)}')


In [ ]:
from collections import Counter

if not signals:
    print('No BIG_DROP signals found.')
else:
    big_drop = [s for s in signals if 'BIG_DROP' in s.get('trigger_types', [])]
    print(f'Signals with BIG_DROP trigger: {len(big_drop)}')

    print('\nTrigger combinations:')
    for triggers, count in sorted(Counter(tuple(sorted(s['trigger_types'])) for s in signals).items()):
        print(f'  {"+".join(triggers):<30} {count}')

    print('\nFirst BIG_DROP signal:')
    import json
    print(json.dumps(big_drop[0], indent=2, default=str))

In [ ]:
# --- Ad-hoc: biggest day-on-day OSA drop for 1st September 2026 -------------
from store_insight.pipeline.stages._1_detect.detectors.osa_big_drop import big_drop

TARGET_DATE = '2026-09-01'

full_osa = (
    session.table(cfg.tables.osa)
    .join(
        session.table(cfg.tables.hierarchy).select(
            F.col('ITEM_CODE'),
            F.col('ITEM_HIERARCHY_2'),
        ).distinct(),
        on='ITEM_CODE',
        how='left',
    )
    .filter(F.col('ITEM_HIERARCHY_2').is_not_null())
    .filter(F.col('DATE') >= F.lit(str(datetime.date.fromisoformat(TARGET_DATE) - datetime.timedelta(days=7))))
    .filter(F.col('DATE') <= F.lit(TARGET_DATE))
    .cache_result()
)

all_changes = big_drop.detect(
    full_osa,
    this_week_end=TARGET_DATE,
    min_actionable_big_drop=0.0,
    min_osa_big_drop_lsv=0.0,
    min_big_drop_positive_rows=0,
    max_actionable_unknown_row_pct=100.0,
)

print(f'Pairs with data on {TARGET_DATE} and previous sales day: {len(all_changes)}')
for s in sorted(all_changes, key=lambda x: x['wow_drift_pp'])[:20]:
    print(
        'store=', s['store_id'],
        'dept=', s['department'],
        'drop=', s['wow_drift_pp'],
        'prev=', s['last_week_osa'],
        'today=', s['this_week_osa'],
        'lsv=', s['this_week_lsv'],
    )


In [ ]:
from store_insight.pipeline.stages._1_detect.detectors.osa_big_drop import big_drop

# Run with quality gates disabled to see how many candidates pass the 5% drop threshold
raw_candidates = big_drop.detect(
    data,
    this_week_end=THIS_WEEK_END,
    min_actionable_big_drop=0.05,
    min_osa_big_drop_lsv=0.0,
    min_big_drop_positive_rows=0,
    max_actionable_unknown_row_pct=100.0,
)

print(f'Candidates with >5pp day-on-day drop (quality gating removed): {len(raw_candidates)}')
for s in sorted(raw_candidates, key=lambda x: x['wow_drift_pp'])[:20]:
    print(
        'store=', s['store_id'],
        'dept=', s['department'],
        'drop=', s['wow_drift_pp'],
        'prev=', s['last_week_osa'],
        'today=', s['this_week_osa'],
        'lsv=', s['this_week_lsv'],
    )


In [ ]:
# --- Investigation: why are big drops being filtered? ------------------------
from store_insight.pipeline.stages._1_detect.detectors.osa_big_drop import big_drop

# Run with all thresholds disabled to see the raw distribution
raw = big_drop.detect(
    data,
    this_week_end=THIS_WEEK_END,
    min_actionable_big_drop=0.0,
    min_osa_big_drop_lsv=0.0,
    min_big_drop_positive_rows=0,
    max_actionable_unknown_row_pct=100.0,
)

print(f'Raw (store, department) pairs: {len(raw)}')
if raw:
    buckets = {}
    for s in raw:
        drop = float(s['wow_drift_pp'] or 0)
        if drop < -10:
            bucket = '< -10pp'
        elif drop < -5:
            bucket = '-10pp to -5pp'
        elif drop < 0:
            bucket = '-5pp to 0pp'
        elif drop == 0:
            bucket = '0pp'
        else:
            bucket = '> 0pp'
        buckets[bucket] = buckets.get(bucket, 0) + 1

    print('Distribution of store-controlled OSA changes:')
    for bucket, count in sorted(buckets.items(), key=lambda x: x[0]):
        print(f'  {bucket:<15} {count}')

    print('\nTop 20 raw drops (including low/no LSV):')
    for s in sorted(raw, key=lambda x: x['wow_drift_pp'])[:20]:
        print(
            'store=', s['store_id'],
            'dept=', s['department'],
            'drop=', s['wow_drift_pp'],
            'prev=', s['last_week_osa'],
            'today=', s['this_week_osa'],
            'store_lsv=', s['this_week_lsv']['store_controlled'] if s['this_week_lsv'] else None,
        )

    big_drops = [s for s in raw if (s['wow_drift_pp'] or 0) < -5]
    print(f'\nPairs with >5pp raw drop: {len(big_drops)}')
    big_drops_sorted_by_lsv = sorted(big_drops, key=lambda x: (x['this_week_lsv'] or {}).get('store_controlled', 0), reverse=True)
    print('Top 20 by store-controlled LSV among >5pp drops:')
    for s in big_drops_sorted_by_lsv[:20]:
        print(
            'store=', s['store_id'],
            'dept=', s['department'],
            'drop=', s['wow_drift_pp'],
            'prev=', s['last_week_osa'],
            'today=', s['this_week_osa'],
            'store_lsv=', s['this_week_lsv']['store_controlled'] if s['this_week_lsv'] else None,
        )
else:
    print('No pairs with data for both days.')


In [ ]:
from store_insight.pipeline.stages._2_score.score import score_all

if signals:
    score_all(
        signals,
        this_week_start=THIS_WEEK_START,
        this_week_end=THIS_WEEK_END,
        osa=osa,
    )
    for s in signals[:5]:
        print(
            f'store={s["store_id"]}  '
            f'dept={s["department"]}  '
            f'score={s.get("score")}  '
            f'projected_cost={s.get("projected_cost")}'
        )
else:
    print('No signals to score.')